In [ ]:
# @title 데이터 로드 및 실행 환경별 경로 설정
# RUN_ENV만 바꾸면 데이터 입력 경로와 모델 저장/로드 경로가 함께 바뀝니다.
RUN_ENV = "kaggle"  # @param ["colab", "kaggle"]

import json
import os
import glob
import pandas as pd

COLAB_BASE_PATH = "/content/drive/MyDrive/AI_Agent_Project"  # @param {type:"string"}
KAGGLE_INPUT_BASE_PATH = "/kaggle/input/datasets/pyeong85732/simmc-csv"  # @param {type:"string"}
COLAB_OUTPUT_DIR = f"{COLAB_BASE_PATH}/Saved_Models"  # @param {type:"string"}
KAGGLE_OUTPUT_DIR = "/kaggle/working/Saved_Models"  # @param {type:"string"}

FILE_NAMES = {
    "origin_train": "Origin_Train_Data_10k.csv",
    "aug_train": "Train_Data_10k_2.csv",
    "val": "Middle_Test_Data.csv",
    "test": "Final_Test_Data.csv",
}

if RUN_ENV not in {"colab", "kaggle"}:
    raise ValueError("RUN_ENV는 'colab' 또는 'kaggle'만 사용할 수 있습니다.")

if RUN_ENV == "colab":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ModuleNotFoundError:
        print("google.colab 모듈이 없습니다. Colab이 아니라면 RUN_ENV='kaggle'로 설정하세요.")
    base_path = COLAB_BASE_PATH
    output_dir = COLAB_OUTPUT_DIR
else:
    base_path = KAGGLE_INPUT_BASE_PATH
    output_dir = KAGGLE_OUTPUT_DIR

os.makedirs(output_dir, exist_ok=True)
save_dir = output_dir

origin_train_path = os.path.join(base_path, FILE_NAMES["origin_train"])
aug_train_path = os.path.join(base_path, FILE_NAMES["aug_train"])
val_path = os.path.join(base_path, FILE_NAMES["val"])
test_path = os.path.join(base_path, FILE_NAMES["test"])
TARGET_DF_PATH = test_path

# 특정 모델 파일을 직접 지정하려면 아래 값을 채우세요. 비워두면 output_dir에서 가장 최근 .pt 파일을 자동 선택합니다.
EXACT_MODEL_PATH = ""  # @param {type:"string"}


def resolve_model_path(preferred_path=None, pattern="*.pt"):
    preferred_path = preferred_path or EXACT_MODEL_PATH
    if preferred_path and os.path.exists(preferred_path):
        return preferred_path

    model_files = glob.glob(os.path.join(output_dir, pattern))
    if pattern != "*.pt":
        model_files.extend(glob.glob(os.path.join(output_dir, "*.pt")))
    model_files = sorted(set(model_files), key=os.path.getmtime)
    if model_files:
        return model_files[-1]

    if preferred_path:
        return preferred_path
    return os.path.join(output_dir, "Advanced_Best_Model.pt")

print(f"실행 환경: {RUN_ENV}")
print(f"데이터 경로: {base_path}")
print(f"모델 저장/로드 경로: {output_dir}")

try:
    df_origin_train = pd.read_csv(origin_train_path)
    df_aug_train = pd.read_csv(aug_train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)

    print("모든 데이터를 성공적으로 로드했습니다.")
    print(f"Origin_Train_Data_10k 크기: {len(df_origin_train)}")
    print(f"Train_Data_10k_2 크기: {len(df_aug_train)}")
    print(f"Middle_Test_Data 크기: {len(val_df)}")
    print(f"Final_Test_Data 크기: {len(test_df)}")
except FileNotFoundError as e:
    print(f"일부 파일을 찾을 수 없습니다: {e}")
    print(f"origin_train_path = {origin_train_path}")
    print(f"aug_train_path    = {aug_train_path}")
    print(f"val_path          = {val_path}")
    print(f"test_path         = {test_path}")
    if RUN_ENV == "kaggle":
        print("Kaggle 오른쪽 Data 탭에서 실제 input 경로를 확인한 뒤 KAGGLE_INPUT_BASE_PATH를 수정하세요.")
    raise



## 1. 데이터 로드

2단계 학습 파이프라인 구축을 위해 **순수 원본 훈련 데이터**(`Origin_Train_Data_10k.csv`)와 **LLM으로 증강된 훈련 데이터**(`Train_Data_10k_2.csv`)를 함께 로드합니다.

또한, 학습 중 조기 종료 및 검증을 위한 별도의 **검증용 데이터**(`Middle_Test_Data.csv`)와 최종 성능 평가를 위한 **테스트용 데이터**(`Final_Test_Data.csv`)도 로드합니다.

## 2. 기본 데이터셋 선택 및 확인

이전과 달리 데이터를 80/20으로 임의 분할하지 않고, 사전에 준비된 훈련(Train), 검증(Middle_Test), 테스트(Final_Test) 세트를 그대로 사용합니다.

여기서는 아래 이어질 기초 전처리(토크나이저 등) 과정에서 기준이 될 임시 훈련 데이터를 하나 선택하고 크기를 확인합니다.
*(실제 3개 데이터셋 비교 학습 및 튜닝은 맨 마지막 파이프라인에서 자동으로 모두 진행되므로, 여기서는 데이터가 잘 로드되는지 확인하는 용도입니다.)*

In [ ]:
# 텍스트 데이터는 'transcript' 열에, 레이블은 'action_id' 열에 있다고 가정합니다.
text_column_name = 'transcript'  # 예시: 'review', 'comment' 등
label_column_name = 'action_id' # 예시: 'sentiment', 'category' 등

# @title 학습할 기본 데이터 확인
# 기초 전처리(토크나이저 등) 확인을 위해 10k 원본 데이터를 기준으로 설정합니다.
SELECTED_TRAIN_DATA = '10k'
train_df = df_origin_train
data_name_str = 'Train_10k'

# 하위 셀 호환성을 위해 df 변수에도 할당
df = train_df

# 검증(val_df) 및 테스트(test_df)는 이미 로드된 독립된 데이터셋을 사용하므로 분할이 불필요합니다.
print(f"기준 훈련 세트({SELECTED_TRAIN_DATA}) 크기: {len(train_df)} 샘플")
print(f"검증 세트(Middle_Test) 크기: {len(val_df)} 샘플")
print(f"테스트 세트(Final_Test) 크기: {len(test_df)} 샘플")

display(train_df.head())


## 3. BERT 모델 훈련을 위한 라이브러리 설치(비활성화)

BERT 모델을 사용하기 위해 `transformers` 라이브러리와 PyTorch를 설치합니다.

In [ ]:
#pip install transformers[torch] accelerate -U

<!--
## 3.1. Numpy 버전 다운그레이드 (선택 사항 및 문제 해결용)
현재는 불필요하여 비활성화되었습니다.
-->

In [ ]:
# Numpy 다운그레이드는 특정 오류 발생 시에만 필요하므로 비활성화(주석 처리) 해두었습니다.
# 런타임 재시작이 번거로우시다면 이 셀은 무시하고 넘어가시면 됩니다.

# !pip install numpy==1.26.4
# print("Numpy 버전 다운그레이드 완료 (문제 해결 목적).")

## 4. 레이블 값 확인 및 재매핑 (필요시)

In [ ]:
# @title
print(f"'action_id' 열의 고유 레이블: {df[label_column_name].unique()}")
print(f"'action_id' 열의 최소 레이블: {df[label_column_name].min()}")
print(f"'action_id' 열의 최대 레이블: {df[label_column_name].max()}")

# 만약 레이블이 0부터 시작하지 않거나 연속적이지 않다면 재매핑 필요
# 예시: 레이블이 [1, 2, 3, 4, 5, 6]이라면 [0, 1, 2, 3, 4, 5]로 매핑해야 함
# 현재 num_labels가 6이므로, 레이블은 0부터 5까지의 정수여야 합니다.
# 만약 최대 레이블 값이 num_labels - 1보다 크다면 이 단계가 필요합니다.

# 현재 데이터프레임의 레이블 값을 확인한 후, 필요하다면 아래 코드를 활성화하여 실행합니다.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[label_column_name] = le.fit_transform(df[label_column_name])
train_df[label_column_name] = le.transform(train_df[label_column_name])
val_df[label_column_name] = le.transform(val_df[label_column_name])
test_df[label_column_name] = le.transform(test_df[label_column_name])

print(f"재매핑 후 'action_id' 열의 고유 레이블: {df[label_column_name].unique()}")
print(f"재매핑 후 'action_id' 열의 최소 레이블: {df[label_column_name].min()}")
print(f"재매핑 후 'action_id' 열의 최대 레이블: {df[label_column_name].max()}")

# 재매핑 후에는 데이터셋과 데이터로더도 다시 생성해야 할 수 있습니다.
# 이 코드 블록을 실행한 후 결과를 보고 다음 단계를 결정하겠습니다.

## 5. 데이터 전처리 및 토큰화

BERT 모델은 텍스트 데이터를 특정 형식으로 처리해야 합니다. `BertTokenizer`를 사용하여 텍스트를 토큰화하고, 모델 입력에 필요한 형식으로 변환합니다.

In [ ]:
# @title
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# AutoTokenizer를 사용하여 bert-base-uncased 로드
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 커스텀 데이터셋 클래스 정의
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 최대 시퀀스 길이 설정 (BERT 모델에 따라 조정)
MAX_LEN = 128

# 훈련, 검증, 테스트 데이터셋 생성
train_dataset = TextDataset(
    texts=train_df[text_column_name].to_numpy(),
    labels=train_df[label_column_name].to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df[text_column_name].to_numpy(),
    labels=val_df[label_column_name].to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df[text_column_name].to_numpy(),
    labels=test_df[label_column_name].to_numpy(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

print(f"훈련 데이터셋 크기: {len(train_dataset)}")
print(f"검증 데이터셋 크기: {len(val_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

## 6. 데이터로더 생성

훈련, 검증 및 테스트 데이터셋을 `DataLoader`에 넣어 배치 단위로 데이터를 효율적으로 처리할 수 있도록 합니다.

In [ ]:
# @title
BATCH_SIZE = 16

train_data_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0  # 멀티프로세싱 오류를 방지하기 위해 num_workers를 0으로 설정
)

val_data_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0 # 멀티프로세싱 오류를 방지하기 위해 num_workers를 0으로 설정
)

test_data_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0 # 멀티프로세싱 오류를 방지하기 위해 num_workers를 0으로 설정
)

print(f"훈련 데이터로더의 배치 수: {len(train_data_loader)}")
print(f"검증 데이터로더의 배치 수: {len(val_data_loader)}")
print(f"테스트 데이터로더의 배치 수: {len(test_data_loader)}")

## 7. BERT 모델 초기화 (전역 설정 확인용)

사전 훈련된 BERT 모델의 구조와 레이블 수가 정상적으로 맞춰지는지 확인하기 위한 초기화 과정입니다.
*(최종 학습 파이프라인에서는 각 데이터셋의 튜닝과 학습 시마다 모델을 새롭게 초기화하여 사용합니다.)*

In [ ]:
# @title
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np

# 클래스 개수 확인
num_labels = df[label_column_name].nunique()
print(f"감지된 고유 레이블 수: {num_labels}")

# BERT 모델 로드 (uncased)
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels,
    output_attentions=False,
    output_hidden_states=False
)

# GPU 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("BERT 모델 로드 및 장치 설정 완료.")

## 8. 옵티마이저 및 스케줄러 (전역 설정 확인용)

모델 훈련을 위한 옵티마이저와 학습률 스케줄러의 기본 설정입니다.
*(최종 파이프라인에서는 Optuna가 찾은 최적화된 하이퍼파라미터로 매번 새롭게 설정됩니다.)*

In [ ]:
# @title
EPOCHS = 4

optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_data_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

loss_fn = torch.nn.CrossEntropyLoss().to(device)

print("옵티마이저, 스케줄러, 손실 함수 설정 완료.")

## 9. 모델 훈련 및 평가 함수 정의

이제 모델을 훈련하고 각 에포크 후에 성능을 평가하는 함수를 정의합니다. `train_epoch` 함수는 훈련 세트에서 모델을 훈련하고, `eval_model` 함수는 검증 세트에서 모델의 성능을 평가합니다.

In [ ]:
# @title
from sklearn.metrics import classification_report

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses = []
    correct_predictions = 0
    total_samples = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        correct_predictions += torch.sum(preds == labels).item()
        total_samples += labels.size(0)

        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions / total_samples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    total_samples = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds == labels).item()
            total_samples += labels.size(0)

            losses.append(loss.item())

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return correct_predictions / total_samples, np.mean(losses), classification_report(all_labels, all_preds, output_dict=True)

print("훈련 및 평가 함수 정의 완료.")

<!-- 삭제된 섹션 (9. 모델 훈련 실행) - 하단 Optuna 튜닝으로 대체됨 -->

<!-- 삭제된 섹션 (10. 모델 저장) - 하단 코드로 대체됨 -->

<!-- 삭제된 섹션 (11. safetensors 형식으로 모델 저장) -->

<!-- 삭제된 섹션 (12. 테스트 세트 평가) - 하단 코드로 대체됨 -->

## 10. 과적합 방지 클래스 정의 (Early Stopping)

데이터 규모별 성능을 최적화하기 위해, 학습률(Learning Rate)과 규제 강도(Weight Decay) 탐색은 최종 파이프라인 내부의 `Optuna`가 담당합니다.

이 셀에서는 불필요한 학습을 조기에 종료하여 과적합을 방지하고 시간을 단축시켜주는 `EarlyStopping` 클래스를 정의합니다.

In [ ]:
try:
    import optuna
except ImportError:
    !pip install optuna
    import optuna

In [ ]:
# @title
import optuna
import copy
import torch
import numpy as np
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Early Stopping 클래스 정의 (아래 파이프라인에서 필수적으로 사용됩니다)
class EarlyStopping:
    def __init__(self, patience=2, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

print("Early Stopping 클래스 정의 완료.")

## 🚀 11. 고도화된 학습 파이프라인 (Advanced R&D Pipeline)

단 한 번의 실행으로 고도화된 데이터셋 학습부터 최종 성능 평가, 그리고 모델 저장까지 자동으로 수행됩니다.

**[고도화 파이프라인 주요 자동화 과정]**

1. **입력 구조 최적화 (단일 발화 학습)**: 데이터 누수(Data Leakage)를 방지하기 위해 시스템 문맥을 제외하고 사용자 발화(`transcript`) 단독으로 커스텀 데이터셋을 구성하여 모델의 실제 추론 능력을 높입니다.
2. **Stratified K-Fold 기반 다중 지표 최적화**: 훈련셋을 `N_FOLDS`-Fold로 나누어 교차 검증을 수행해 단일 검증셋에 대한 과적합을 방지합니다. Optuna는 Accuracy와 Macro F1의 가중 합산(평균) 값을 최대화하는 방향으로 최적의 하이퍼파라미터(`lr`, `weight_decay`, `dropout_rate`)를 자동 탐색합니다.
3. **최종 모델 학습 및 조기 종료 (Early Stopping)**: 탐색된 최적 파라미터로 본 학습을 진행하며, 성능 향상이 없을 경우 학습을 조기 종료하여 시간을 단축하고 일반화 성능을 확보합니다.
4. **스마트 체크포인트 저장 및 평가**: 최종 테스트 세트를 이용해 모델을 평가한 뒤, 구글 드라이브에 `Advanced_Best_Model_Acc_0.xxxx_F1_0.xxxx.pt`와 같이 파일명에 성능 지표가 포함된 형태로 자동 저장하고 요약 리포트를 출력합니다.

In [ ]:
# @title
import os
import copy
import torch
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# --- 1. 고도화된 데이터셋 정의 (문맥 제외, 사용자 발화 단독 학습) ---
class AdvancedTextDataset(Dataset):
    def __init__(self, system_texts, user_texts, labels, tokenizer, max_len):
        self.system_texts = system_texts
        self.user_texts = user_texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        # 데이터 누수 방지를 위해 system_transcript를 제외하고 사용자 발화만 사용
        usr_text = str(self.user_texts[item]) if pd.notna(self.user_texts[item]) else ""

        encoding = self.tokenizer(
            usr_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'token_type_ids': encoding['token_type_ids'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# --- 2. 훈련/평가 함수 업데이트 (token_type_ids 추가) ---
def advanced_train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses, preds_list, labels_list = [], [], []

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)
        loss = outputs.loss

        preds = torch.argmax(outputs.logits, dim=1)
        losses.append(loss.item())
        preds_list.extend(preds.cpu().numpy())
        labels_list.extend(labels.cpu().numpy())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return accuracy_score(labels_list, preds_list), np.mean(losses)

def advanced_eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses, preds_list, labels_list = [], [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)
            loss = outputs.loss

            preds = torch.argmax(outputs.logits, dim=1)
            losses.append(loss.item())
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_list, preds_list)
    macro_f1 = f1_score(labels_list, preds_list, average='macro')
    return acc, macro_f1, np.mean(losses), classification_report(labels_list, preds_list, output_dict=True)

# --- 3. 고도화 파이프라인 메인 함수 ---
def run_advanced_pipeline(df_train, df_test, tokenizer, device, num_labels, n_trials=5, max_epochs=5):
    print("\n🚀 [고도화 버전] Stratified K-Fold & Multi-Metric Optuna 파이프라인 시작...")

    # 첫 셀에서 설정한 output_dir에 모델을 저장합니다.
    os.makedirs(output_dir, exist_ok=True)

    sys_col = 'system_transcript' if 'system_transcript' in df_train.columns else 'transcript'
    usr_col = 'transcript'
    lbl_col = 'action_id'

    def objective(trial):
        lr = trial.suggest_float("lr", 1.8e-5, 2.6e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.01, 0.06, log=True)
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.4)

        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        fold_scores = []

        train_texts_sys = df_train[sys_col].to_numpy()
        train_texts_usr = df_train[usr_col].to_numpy()
        train_labels = df_train[lbl_col].to_numpy()

        for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts_usr, train_labels)):
            print(f"\n--- [Trial {trial.number}] Fold {fold+1}/{N_FOLDS} ---")

            train_ds = AdvancedTextDataset(train_texts_sys[train_idx], train_texts_usr[train_idx], train_labels[train_idx], tokenizer, MAX_LEN)
            val_ds = AdvancedTextDataset(train_texts_sys[val_idx], train_texts_usr[val_idx], train_labels[val_idx], tokenizer, MAX_LEN)

            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=16)

            config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
            config.hidden_dropout_prob = dropout_rate
            config.attention_probs_dropout_prob = dropout_rate
            config.classifier_dropout = dropout_rate

            model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

            no_decay = ['bias', 'LayerNorm.weight']
            optimizer_grouped_parameters = [
                {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': weight_decay},
                {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
            ]
            optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

            epochs = 4
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*epochs)

            best_fold_score = 0
            early_stopping_counter = 0

            for epoch in range(epochs):
                advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)
                val_acc, val_macro_f1, val_loss, _ = advanced_eval_model(model, val_loader, torch.nn.CrossEntropyLoss(), device)

                combined_score = (val_acc + val_macro_f1) / 2.0

                if combined_score > best_fold_score:
                    best_fold_score = combined_score
                    early_stopping_counter = 0
                else:
                    early_stopping_counter += 1

                if early_stopping_counter >= 2:
                    break

            fold_scores.append(best_fold_score)
            break

        return np.mean(fold_scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"\n✅ 최적 하이퍼파라미터 탐색 완료: {study.best_params}")

    print("🚀 전체 훈련셋 대상 최종 모델 학습 시작...")
    best_lr = study.best_params['lr']
    best_wd = study.best_params['weight_decay']
    best_dropout = study.best_params['dropout_rate']

    train_ds = AdvancedTextDataset(df_train[sys_col].to_numpy(), df_train[usr_col].to_numpy(), df_train[lbl_col].to_numpy(), tokenizer, MAX_LEN)
    test_ds = AdvancedTextDataset(df_test[sys_col].to_numpy(), df_test[usr_col].to_numpy(), df_test[lbl_col].to_numpy(), tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16)

    config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
    config.hidden_dropout_prob = best_dropout
    config.attention_probs_dropout_prob = best_dropout
    config.classifier_dropout = best_dropout

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': best_wd},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=best_lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*max_epochs)

    best_combined_score = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stopping_counter = 0

    for epoch in range(max_epochs):
        train_acc, train_loss = advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)
        test_acc, test_macro_f1, test_loss, report = advanced_eval_model(model, test_loader, torch.nn.CrossEntropyLoss(), device)

        combined_score = (test_acc + test_macro_f1) / 2.0
        print(f"Epoch {epoch+1}/{max_epochs} | Train Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}, Test Macro F1: {test_macro_f1:.4f}")

        if combined_score > best_combined_score:
            best_combined_score = combined_score
            best_model_wts = copy.deepcopy(model.state_dict())
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1

        if early_stopping_counter >= 2:
            print(f"⚡ 조기 종료 발동 (Epoch {epoch+1})")
            break

# base_path 대신 첫 셀의 output_dir을 사용합니다.
    save_path = os.path.join(output_dir, "Advanced_Best_Model.pt")
    torch.save(best_model_wts, save_path)
    print(f"💾 최고 성능 모델 저장 완료: {save_path}")


### 11-1. 고도화된 Dataset 및 훈련/평가 함수 정의

데이터 누수 방지를 위해 시스템 문맥(Context)을 제외하고 단일 발화로 구성된 커스텀 데이터셋과 훈련/평가 함수를 정의합니다.

In [ ]:
# @title
import os
import copy
import torch
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# --- 1. 고도화된 데이터셋 정의 (문맥 제외, 사용자 발화 단독) ---
class AdvancedTextDataset(Dataset):
    def __init__(self, system_texts, user_texts, labels, tokenizer, max_len):
        self.system_texts = system_texts
        self.user_texts = user_texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, item):
        usr_text = str(self.user_texts[item]) if pd.notna(self.user_texts[item]) else ""

        encoding = self.tokenizer(
            usr_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'token_type_ids': encoding['token_type_ids'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# --- 2. 훈련/평가 함수 ---
def advanced_train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses, preds_list, labels_list = [], [], []

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)
        loss = outputs.loss

        preds = torch.argmax(outputs.logits, dim=1)
        losses.append(loss.item())
        preds_list.extend(preds.cpu().numpy())
        labels_list.extend(labels.cpu().numpy())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return accuracy_score(labels_list, preds_list), np.mean(losses)

def advanced_eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses, preds_list, labels_list = [], [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)
            loss = outputs.loss

            preds = torch.argmax(outputs.logits, dim=1)
            losses.append(loss.item())
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_list, preds_list)
    macro_f1 = f1_score(labels_list, preds_list, average='macro')
    return acc, macro_f1, np.mean(losses), classification_report(labels_list, preds_list, output_dict=True)

# --- 3. 고도화 파이프라인 메인 함수 ---
def run_advanced_pipeline(df_train, df_test, tokenizer, device, num_labels, n_trials=5, max_epochs=5, n_folds=5):
    print(f"\n🚀 [고도화 버전] Stratified {n_folds}-Fold & Multi-Metric Optuna 파이프라인 시작...")

    # 첫 셀에서 설정한 output_dir에 모델을 저장합니다.
    os.makedirs(output_dir, exist_ok=True)

    sys_col = 'system_transcript' if 'system_transcript' in df_train.columns else 'transcript'
    usr_col = 'transcript'
    lbl_col = 'action_id'

    def objective(trial):
        lr = trial.suggest_float("lr", 1.8e-5, 2.6e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.01, 0.06, log=True)
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.4)

        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
        fold_scores = []

        train_texts_sys = df_train[sys_col].to_numpy()
        train_texts_usr = df_train[usr_col].to_numpy()
        train_labels = df_train[lbl_col].to_numpy()

        for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts_usr, train_labels)):
            print(f"\n--- [Trial {trial.number}] Fold {fold+1}/{n_folds} ---")

            train_ds = AdvancedTextDataset(train_texts_sys[train_idx], train_texts_usr[train_idx], train_labels[train_idx], tokenizer, MAX_LEN)
            val_ds = AdvancedTextDataset(train_texts_sys[val_idx], train_texts_usr[val_idx], train_labels[val_idx], tokenizer, MAX_LEN)

            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=16)

            config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
            config.hidden_dropout_prob = dropout_rate
            config.attention_probs_dropout_prob = dropout_rate
            config.classifier_dropout = dropout_rate

            model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

            no_decay = ['bias', 'LayerNorm.weight']
            optimizer_grouped_parameters = [
                {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': weight_decay},
                {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
            ]
            optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

            epochs = 4
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*epochs)

            best_fold_score = 0
            early_stopping_counter = 0

            for epoch in range(epochs):
                advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)
                val_acc, val_macro_f1, val_loss, _ = advanced_eval_model(model, val_loader, torch.nn.CrossEntropyLoss(), device)

                combined_score = (val_acc + val_macro_f1) / 2.0

                if combined_score > best_fold_score:
                    best_fold_score = combined_score
                    early_stopping_counter = 0
                else:
                    early_stopping_counter += 1

                if early_stopping_counter >= 2:
                    break

            fold_scores.append(best_fold_score)
            break

        return np.mean(fold_scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print(f"\n✅ 최적 하이퍼파라미터 탐색 완료: {study.best_params}")

    print("🚀 전체 훈련셋 대상 최종 모델 학습 시작...")
    best_lr = study.best_params['lr']
    best_wd = study.best_params['weight_decay']
    best_dropout = study.best_params['dropout_rate']

    train_ds = AdvancedTextDataset(df_train[sys_col].to_numpy(), df_train[usr_col].to_numpy(), df_train[lbl_col].to_numpy(), tokenizer, MAX_LEN)
    test_ds = AdvancedTextDataset(df_test[sys_col].to_numpy(), df_test[usr_col].to_numpy(), df_test[lbl_col].to_numpy(), tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16)

    config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
    config.hidden_dropout_prob = best_dropout
    config.attention_probs_dropout_prob = best_dropout
    config.classifier_dropout = best_dropout

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': best_wd},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=best_lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*max_epochs)

    best_combined_score = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stopping_counter = 0

    for epoch in range(max_epochs):
        train_acc, train_loss = advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)
        test_acc, test_macro_f1, test_loss, report = advanced_eval_model(model, test_loader, torch.nn.CrossEntropyLoss(), device)

        combined_score = (test_acc + test_macro_f1) / 2.0
        print(f"Epoch {epoch+1}/{max_epochs} | Train Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}, Test Macro F1: {test_macro_f1:.4f}")

        if combined_score > best_combined_score:
            best_combined_score = combined_score
            best_model_wts = copy.deepcopy(model.state_dict())
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1

        if early_stopping_counter >= 2:
            print(f"⚡ 조기 종료 발동 (Epoch {epoch+1})")
            break

# base_path 대신 첫 셀의 output_dir을 사용합니다.
    save_path = os.path.join(output_dir, "Advanced_Best_Model.pt")
    torch.save(best_model_wts, save_path)
    print(f"💾 최고 성능 모델 저장 완료: {save_path}")


### 11-2. 고도화 파이프라인 핵심 함수 정의 (모델 저장 시 성능 기록)

Optuna 튜닝과 최종 학습, 모델 저장을 수행하는 메인 파이프라인 함수입니다.

In [ ]:
# @title 2단계 전략적 학습 파이프라인 메인 함수 정의
# --- 3. 고도화 파이프라인 메인 함수 ---
def run_2stage_pipeline(df_origin, df_aug, df_val, df_test, tokenizer, device, num_labels, n_trials=5, max_epochs=5, n_folds=5,fast_test=False):
    print("\n🚀 [1단계] 원본 기반 하이퍼파라미터 최적화 (Optuna) 시작...")
    # 텍스트는 오직 transcript 컬럼만 사용
# [수정] 모델 저장용 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)
    
    usr_col = 'transcript'
    lbl_col = 'action_id'

    # 1단계 입력 데이터 구축: 원본 훈련 데이터 + Middle_Test_Data 병합
    df_stage1 = pd.concat([df_origin, df_val], ignore_index=True)
    texts_stage1 = df_stage1[usr_col].to_numpy()
    labels_stage1 = df_stage1[lbl_col].to_numpy()

    def objective(trial):
        # 탐색 범위 설정
        lr = trial.suggest_float("lr", 1.8e-5, 2.6e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.01, 0.06, log=True)
        dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.4)

        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
        fold_scores = []

        # 계층화 K-Fold 교차 검증
        for fold, (train_idx, val_idx) in enumerate(skf.split(texts_stage1, labels_stage1)):
            print(f"\n--- [Trial {trial.number}] Fold {fold+1}/{n_folds} ---")

            # 시스템 발화 배제용 빈 배열을 넣어 AdvancedTextDataset 호환성을 유지
            empty_sys_train = [""] * len(train_idx)
            empty_sys_val = [""] * len(val_idx)

            train_ds = AdvancedTextDataset(empty_sys_train, texts_stage1[train_idx], labels_stage1[train_idx], tokenizer, MAX_LEN)
            fold_val_ds = AdvancedTextDataset(empty_sys_val, texts_stage1[val_idx], labels_stage1[val_idx], tokenizer, MAX_LEN)

            train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
            fold_val_loader = DataLoader(fold_val_ds, batch_size=16)

            config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
            config.hidden_dropout_prob = dropout_rate
            config.attention_probs_dropout_prob = dropout_rate
            config.classifier_dropout = dropout_rate

            model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

            no_decay = ['bias', 'LayerNorm.weight']
            optimizer_grouped_parameters = [
                {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': weight_decay},
                {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
            ]
            optimizer = AdamW(optimizer_grouped_parameters, lr=lr)

            epochs = 3 # Fold 별 빠른 탐색을 위해 3으로 조정
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*epochs)

            best_fold_score = 0
            early_stopping_counter = 0

            for epoch in range(epochs):
                advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)
                val_acc, val_macro_f1, val_loss, _ = advanced_eval_model(model, fold_val_loader, torch.nn.CrossEntropyLoss(), device)

                # 최적화 목표: 평균 Accuracy와 Macro F1의 가중 합산
                combined_score = (val_acc + val_macro_f1) / 2.0

                if combined_score > best_fold_score:
                    best_fold_score = combined_score
                    early_stopping_counter = 0
                else:
                    early_stopping_counter += 1

                if early_stopping_counter >= 2:
                    print(f"⚡ Fold 조기 종료 (Epoch {epoch+1})")
                    break

            fold_scores.append(best_fold_score)
            # 빠른 전체 파이프라인 테스트를 위해 1개 Fold만 진행
            if fast_test:
                print("⚡ 빠른 테스트 모드: 1 Fold만 수행하고 교차 검증을 종료합니다.")
                break

        return np.mean(fold_scores)

    # 1단계 Optuna 실행
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"\n✅ [1단계 완료] 최적 하이퍼파라미터 탐색 완료: {study.best_params}")

    # ---------------------------------------------------------
    print("\n🚀 [2단계] 증강 데이터 포함 최종 모델 학습 시작...")

    best_lr = study.best_params['lr']
    best_wd = study.best_params['weight_decay']
    best_dropout = study.best_params['dropout_rate']

    # 2단계 입력 데이터 구축: 훈련셋 = 원본 + 증강 데이터
    df_train_final = pd.concat([df_origin, df_aug], ignore_index=True)
    print(f"최종 훈련 데이터 크기(원본+증강): {len(df_train_final)}")

    train_ds = AdvancedTextDataset([""]*len(df_train_final), df_train_final[usr_col].to_numpy(), df_train_final[lbl_col].to_numpy(), tokenizer, MAX_LEN)
    val_ds = AdvancedTextDataset([""]*len(df_val), df_val[usr_col].to_numpy(), df_val[lbl_col].to_numpy(), tokenizer, MAX_LEN)
    test_ds = AdvancedTextDataset([""]*len(df_test), df_test[usr_col].to_numpy(), df_test[lbl_col].to_numpy(), tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)
    test_loader = DataLoader(test_ds, batch_size=16)

    # 모델 및 옵티마이저 최적 파라미터로 재설정
    config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
    config.hidden_dropout_prob = best_dropout
    config.attention_probs_dropout_prob = best_dropout
    config.classifier_dropout = best_dropout

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)

    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': best_wd},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in ['bias', 'LayerNorm.weight'])], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=best_lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*max_epochs)

    best_val_macro_f1 = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stopping_counter = 0

    for epoch in range(max_epochs):
        train_acc, train_loss = advanced_train_epoch(model, train_loader, torch.nn.CrossEntropyLoss(), optimizer, device, scheduler)

        # 오직 Middle_Test_Data 원본으로만 검증
        val_acc, val_macro_f1, val_loss, _ = advanced_eval_model(model, val_loader, torch.nn.CrossEntropyLoss(), device)
        print(f"Epoch {epoch+1}/{max_epochs} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}, Val Macro F1: {val_macro_f1:.4f}")

        # 저장 기준: 검증 Macro F1이 가장 높을 때
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1

        if early_stopping_counter >= 2:
            print(f"⚡ 최종 학습 조기 종료 발동 (Epoch {epoch+1})")
            break

    # 결과 변수가 포함된 모델 저장 로직
# [수정] 2단계 학습 후 최종 모델 저장 경로 변경
    # 첫 셀에서 설정한 실행 환경별 output_dir에 저장합니다.
    save_path = os.path.join(output_dir, f"Stage2_Best_Model_ValF1_{best_val_macro_f1:.4f}.pt")
    torch.save(best_model_wts, save_path)
    print(f"\n💾 [2단계 완료] 최고 성능(Val F1) 모델 저장 완료: {save_path}")

    # ---------------------------------------------------------
    # 최종 평가: Final_Test_Data 사용
    print("\n🔍 [최종 평가] 베스트 모델 로드 및 테스트 데이터(Final_Test_Data) 평가 중...")
    model.load_state_dict(best_model_wts)
    test_acc, test_macro_f1, test_loss, report = advanced_eval_model(model, test_loader, torch.nn.CrossEntropyLoss(), device)
    print(f"✅ 최종 Test Acc: {test_acc:.4f}, 최종 Test Macro F1: {test_macro_f1:.4f}")

    return {
        'Best_Acc': test_acc,
        'Best_F1': test_macro_f1,
        'Saved_Path': save_path
    }


### 11-3. 고도화 파이프라인 실행 및 결과 확인

위에서 정의한 파이프라인을 실행합니다.

**⏱️ 예상 총 훈련 시간 안내 (기본 설정 기준)**
- 1 Fold 당 약 18분, 2단계 최종 학습 약 30분 소요 예상 (Optuna 3회 탐색 기준)
- **빠른 테스트 모드 (ON)**: 약 1시간 24분 (1단계 54분 + 2단계 30분)
- **전체 교차 검증 (OFF)**: 약 5시간 (1단계 270분 + 2단계 30분)

kaggle용 설정(12시간 기준)
1fold 약 1시간 6분
3~4 trial, 2 fold

In [ ]:
# @title 고도화 파이프라인 하이퍼파라미터 설정 및 실행

# Optuna에서 탐색할 트라이얼(Trial) 횟수
OPTUNA_TRIALS = 3 # @param {type:"slider", min:1, max:20, step:1}
# 각 Trial 및 최종 학습 시 최대 에포크 수
MAX_EPOCHS = 5 # @param {type:"slider", min:3, max:10, step:1}

N_FOLDS = 2  # @param {type:"integer"}

# 빠른 테스트 모드 (1-Fold만 수행하여 시간 단축)
FAST_TEST = False # @param {type:"boolean"}

print(f"설정된 Optuna 트라이얼 횟수: {OPTUNA_TRIALS}")
print(f"설정된 최대 에포크 수: {MAX_EPOCHS}")
print(f"기본 교차 검증 폴드 수: {N_FOLDS}")
print(f"빠른 테스트 모드: {'활성화 (1-Fold만 수행)' if FAST_TEST else f'비활성화 (전체 {N_FOLDS}-Fold 수행)'}")
print("-" * 50)

# 1 Fold 당 예상 소요 시간(분) 및 2단계 학습 예상 시간(분) 설정
fold_time = 18 # @param {type:"integer"}
stage2_mins = 30 # @param {type:"integer"}

# 예상 시간 계산
# FAST_TEST ON (1-Fold)
stage1_mins_fast = OPTUNA_TRIALS * 1 * fold_time
total_mins_fast = stage1_mins_fast + stage2_mins

# FAST_TEST OFF (N_FOLDS-Fold)
stage1_mins_full = OPTUNA_TRIALS * N_FOLDS * fold_time
total_mins_full = stage1_mins_full + stage2_mins

print(f"⏱️ 예상 총 훈련 시간 안내 (설정값: 1 Fold = {fold_time}분, 2단계 = {stage2_mins}분 기준)")
print(f"  - 빠른 테스트 모드 [ON]  : 약 {total_mins_fast // 60}시간 {total_mins_fast % 60}분 (1단계 {stage1_mins_fast}분 + 2단계 약 {stage2_mins}분)")
print(f"  - 빠른 테스트 모드 [OFF] : 약 {total_mins_full // 60}시간 {total_mins_full % 60}분 (1단계 {stage1_mins_full}분 + 2단계 약 {stage2_mins}분)")
print(f"  👉 현재 설정 기준 예상 시간: {'[빠른 테스트 ON]' if FAST_TEST else '[빠른 테스트 OFF]'} 적용됨")
print("-" * 50)

# 파이프라인 실행 및 결과 받기
final_results = run_2stage_pipeline(
    df_origin=df_origin_train,  # 10k 순수 원본 훈련 데이터
    df_aug=df_aug_train,        # 10k 원본 기반 LLM 증강 데이터
    df_val=val_df,              # Middle_Test_Data (튜닝/검증 조기종료용)
    df_test=test_df,            # Final_Test_Data (최종 평가용)
    tokenizer=tokenizer,
    device=device,
    num_labels=num_labels,
    n_trials=OPTUNA_TRIALS,
    max_epochs=MAX_EPOCHS,
    n_folds=N_FOLDS,
    fast_test=FAST_TEST
)

print("\n================ 🏆 2단계 파이프라인 최종 모델 성능 요약 ================")
print(f"테스트 Accuracy: {final_results['Best_Acc']:.4f}")
print(f"테스트 Macro F1: {final_results['Best_F1']:.4f}")
print(f"저장된 경로: {final_results['Saved_Path']}")

## 12. 저장된 고도화 모델 불러오기 및 평가 (선택 사항)

나중에 코랩 런타임을 다시 시작했을 때, 다시 학습할 필요 없이 구글 드라이브에 저장된 `Advanced_Best_Model_...pt` 파일을 불러와서 최종 테스트 세트로 평가해 보는 코드입니다.

In [ ]:
# @title 저장된 가장 최근 모델 로드 및 평가
import os
import glob
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

# 첫 셀에서 설정한 모델 저장 경로를 사용합니다.
save_dir = output_dir
model_files = glob.glob(os.path.join(save_dir, "Advanced_Best_Model*.pt"))

if not model_files:
    print("❌ 저장된 고도화 모델이 없습니다. 11번 파이프라인을 먼저 실행해 주세요.")
else:
    # 가장 최근에 생성/수정된 모델 선택
    latest_model_path = max(model_files, key=os.path.getmtime)
    print(f"📂 모델 파일 로드 중: {latest_model_path}")

    # 2. 모델 구조 초기화 및 가중치 로드
    config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_labels)
    loaded_model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        config=config,
        ignore_mismatched_sizes=True
    ).to(device)

    loaded_model.load_state_dict(torch.load(latest_model_path, map_location=device))
    print("✅ 가중치 로드 완료! 평가 진행 중...")

    # 3. 테스트 데이터 로더 재구성 (문맥 결합 버전)
    sys_col = 'system_transcript' if 'system_transcript' in test_df.columns else 'transcript'
    usr_col = 'transcript'
    lbl_col = 'action_id'

    test_ds = AdvancedTextDataset(
        test_df[sys_col].to_numpy(),
        test_df[usr_col].to_numpy(),
        test_df[lbl_col].to_numpy(),
        tokenizer,
        MAX_LEN
    )
    test_loader = DataLoader(test_ds, batch_size=16)

    # 4. 평가 수행
    test_acc, test_macro_f1, test_loss, report = advanced_eval_model(
        loaded_model,
        test_loader,
        torch.nn.CrossEntropyLoss(),
        device
    )

    print("\n================ 🏆 불러온 모델 최종 성능 ================")
    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"Test Macro F1 : {test_macro_f1:.4f}")
    print("===========================================================")


In [ ]:
# --- 저장된 고도화 모델 로드 및 평가 (캐글 최적화 완료) ---
import os
import glob
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. 첫 셀에서 설정한 저장 경로에서 고도화 모델 찾기
# 이전 단계에서 저장했던 Saved_Models 폴더를 탐색합니다.
save_dir = output_dir
model_files = glob.glob(os.path.join(save_dir, "*.pt"))

if not model_files:
    print(f"❌ 저장된 고도화 모델이 없습니다. 경로 확인: {save_dir}")
else:
    # 가장 최근에 생성/수정된 모델 선택
    latest_model_path = max(model_files, key=os.path.getmtime)
    print(f"📂 모델 파일 로드 중: {latest_model_path}")

    # 2. 모델 구조 초기화 및 가중치 로드
    # 이전 셀에서 정의한 AdvancedTextDataset 클래스를 그대로 사용합니다.
    config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=7)
    loaded_model = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        config=config,
        ignore_mismatched_sizes=True
    ).to(device)

    loaded_model.load_state_dict(torch.load(latest_model_path, map_location=device))
    print("✅ 가중치 로드 완료! 평가 진행 중...")

    # 3. 테스트 데이터 로더 구성
    # test_df가 메모리에 로드되어 있어야 합니다.
    sys_col = 'system_transcript' if 'system_transcript' in test_df.columns else 'transcript'
    usr_col = 'transcript'
    lbl_col = 'action_id'

    test_ds = AdvancedTextDataset(
        test_df[sys_col].to_numpy(),
        test_df[usr_col].to_numpy(),
        test_df[lbl_col].to_numpy(),
        tokenizer,
        MAX_LEN
    )
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    # 4. 평가 수행
    loaded_model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = loaded_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    class_names = ['SEARCH_G', 'CHECK_IN', 'REFINE_SE', 'INFORM_P', 'ADD_TO_C', 'ORDER_PUR', 'CHITCHAT']

    print("\n" + "="*25 + " 🏆 모델 최종 성능 리포트 " + "="*25)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    
    # 5. 혼동 행렬 시각화
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()


In [ ]:
# @title 결과 샘플링 시각화
NUM_CORRECT_PER_LABEL = 1 # @param {type:"slider", min:1, max:5, step:1}
NUM_ERROR_PER_LABEL = 1 # @param {type:"slider", min:1, max:5, step:1}

import os
import torch
import pandas as pd
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
from torch.utils.data import DataLoader, Dataset

action_to_id = {
    'SEARCH_GET': 0,
    'CHECK_INFO': 1,
    'REFINE_SEARCH': 2,
    'INFORM_PREFERENCE': 3,
    'COMPARE': 4,
    'ADD_TO_CART': 5,
    'CHITCHAT_ETC': 6
}
id_to_action = {v: k for k, v in action_to_id.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 1. 캐글 모델 경로 자동 탐색 (가장 최근 파일)
save_dir = output_dir
model_files = glob.glob(os.path.join(save_dir, "Advanced_Best_Model*.pt"))
NUM_LABELS = 7

TARGET_DF_PATH = test_path
EXACT_MODEL_PATH = resolve_model_path(pattern="Advanced_Best_Model*.pt")
df = pd.read_csv(TARGET_DF_PATH)

config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=NUM_LABELS)
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config, ignore_mismatched_sizes=True).to(device)
model.load_state_dict(torch.load(EXACT_MODEL_PATH, map_location=device))
model.eval()

class AdvancedTextDataset(Dataset):
    def __init__(self, sys_texts, usr_texts, labels, tokenizer, max_len):
        self.sys_texts = sys_texts
        self.usr_texts = usr_texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.usr_texts)

    def __getitem__(self, idx):
        usr_text = str(self.usr_texts[idx]) if pd.notna(self.usr_texts[idx]) else ""
        label = self.labels[idx]

        inputs = self.tokenizer(
            usr_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

sys_col = 'system_transcript' if 'system_transcript' in df.columns else 'transcript'
usr_col = 'transcript'
lbl_col = 'action_id'

dataset = AdvancedTextDataset(df[sys_col].to_numpy(), df[usr_col].to_numpy(), df[lbl_col].to_numpy(), tokenizer, MAX_LEN)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

all_preds = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

df['predicted_action_id'] = all_preds
df['predicted_act'] = df['predicted_action_id'].map(id_to_action)

if 'act' not in df.columns:
    df['act'] = df['action_id'].map(id_to_action)

df['is_correct'] = df['action_id'] == df['predicted_action_id']

display_cols = ['transcript', 'system_transcript', 'act', 'action_id', 'predicted_act', 'predicted_action_id', 'is_correct']
if 'system_transcript' not in df.columns:
    display_cols.remove('system_transcript')

result_df = df[display_cols]

print(f"=== ✅ 정답 예측 샘플 (각 레이블별 최대 {NUM_CORRECT_PER_LABEL}개씩) ===")
correct_list = []
for label_id, label_name in id_to_action.items():
    corr_df = result_df[(result_df['is_correct'] == True) & (result_df['action_id'] == label_id)]
    if not corr_df.empty:
        corr_sample = corr_df.sample(n=min(NUM_CORRECT_PER_LABEL, len(corr_df)), random_state=42).copy()
        correct_list.append(corr_sample)

if correct_list:
    correct_samples = pd.concat(correct_list)
    display(correct_samples)
else:
    print("정답이 없습니다.")

print(f"\n=== ❌ 오답 예측 샘플 (각 레이블별 FP, FN 최대 {NUM_ERROR_PER_LABEL}개씩) ===")
fp_fn_list = []
for label_id, label_name in id_to_action.items():
    fp_df = result_df[(result_df['predicted_action_id'] == label_id) & (result_df['action_id'] != label_id)]
    if not fp_df.empty:
        fp_sample = fp_df.sample(n=min(NUM_ERROR_PER_LABEL, len(fp_df)), random_state=42).copy()
        fp_sample['Error_Type'] = f"FP (Predicted {label_name})"
        fp_fn_list.append(fp_sample)

    fn_df = result_df[(result_df['action_id'] == label_id) & (result_df['predicted_action_id'] != label_id)]
    if not fn_df.empty:
        fn_sample = fn_df.sample(n=min(NUM_ERROR_PER_LABEL, len(fn_df)), random_state=42).copy()
        fn_sample['Error_Type'] = f"FN (Missed {label_name})"
        fp_fn_list.append(fn_sample)

if fp_fn_list:
    incorrect_samples = pd.concat(fp_fn_list)
    cols = ['Error_Type'] + [c for c in incorrect_samples.columns if c != 'Error_Type']
    display(incorrect_samples[cols])
else:
    print("오답이 없습니다.")
